# Project 11 — Tiny GPT (tokenizer → embeddings → blocks → LM head)

```
text -> char-tokenizer -> token-ids -> tok-embed + pos-embed -> [LN -> causal-MHA -> +res -> LN -> MLP -> +res]xN -> LN -> logits(V) -> CE-next-token
```

**Goal:** train a real (small) autoregressive transformer on TinyShakespeare: build the
char tokenizer, the GPT blocks from P10's MHA, the next-token training loop, and sample
new text. Show loss dropping below the uniform baseline and ablate positional encodings.

**Learning objectives:** (1) tokenize text and batch (x,y)-shifted pairs; (2) assemble a GPT
(embeddings, blocks, tied/untied head); (3) run the LM training loop (CE + AdamW);
(4) sample greedily/with temperature; (5) measure tok/s; (6) explain why order matters.


In [ ]:
import os # need os to create results/ and check cwd
import random # need random to seed stdlib RNG
import time # need time to measure tok/s and epoch timing
import csv # need csv to write loss tables
import math # need math for loss-baseline ln(V) and perplexity exp()
import urllib.request # need urllib to try downloading TinyShakespeare (fallback offline)
import numpy as np # need numpy for data splits and batch indexing
import torch # need torch for the GPT model and training loop
import torch.nn.functional as F # need F for cross-entropy and softmax sampling
import matplotlib # need matplotlib for loss curves
matplotlib.use('Agg') # force headless backend for PNG output
import matplotlib.pyplot as plt # need pyplot for the loss-curve figure
SEED = 0 # fixed seed: same init, same batches, same samples
random.seed(SEED) # seed stdlib
np.random.seed(SEED) # seed numpy
torch.manual_seed(SEED) # seed torch
torch.set_num_threads(max(1, os.cpu_count() // 2)) # cap threads: stable timing on shared boxes
os.makedirs('./results', exist_ok=True) # create results/ for CSVs/PNGs/samples
DEV = 'cuda' if torch.cuda.is_available() else 'cpu' # GPU if present else CPU
print(f'device={DEV} torch={torch.__version__}') # env line for the README


In [ ]:
# ---- data: TinyShakespeare download with offline synthetic fallback ----
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt' # canonical tiny-shakespeare source
MAX_CHARS = 120000 # cap corpus: trains fast on CPU yet keeps Shakespeare flavor
def synthetic_text(n): # deterministic fallback when offline: pseudo-early-modern loop
    words = ['to', 'be', 'thou', 'thee', 'king', 'queen', 'night', 'day', 'love', 'death', 'heart', 'lord', 'sweet', 'fair', 'dark', 'light', 'shall', 'will', 'not', 'and'] # small vocab pool
    rng = random.Random(SEED) # seeded: identical fallback every run
    out = [] # collect words
    while sum(len(w) + 1 for w in out) < n: # fill until n chars
        out.append(rng.choice(words)) # pick next word
        if len(out) % 9 == 0: # every 9 words...
            out.append('\n') # ...line break like verse
    return ' '.join(out)[:n] # join and trim to exactly n
try: # attempt download first (needs network + ~5MB)
    with urllib.request.urlopen(URL, timeout=15) as r: # 15s timeout: never hang headless runs
        text = r.read().decode('utf-8')[:MAX_CHARS] # decode and cap length
    SRC = 'tinyshakespeare' # record real-data provenance
except Exception as e: # offline or blocked: fall back
    print(f'download failed ({e}); using synthetic fallback') # say why
    text = synthetic_text(MAX_CHARS) # deterministic substitute
    SRC = 'synthetic-fallback' # record fallback provenance
chars = sorted(set(text)) # vocab = sorted unique chars (char-level tokenizer)
V = len(chars) # vocab size (~65 for Shakespeare)
stoi = {c: i for i, c in enumerate(chars)} # char -> id map
itos = {i: c for c, i in stoi.items()} # id -> char map
encode = lambda s: [stoi[c] for c in s] # tokenizer: string -> id list
decode = lambda ids: ''.join(itos[i] for i in ids) # detokenizer: id list -> string
data = torch.tensor(encode(text), dtype=torch.long) # full corpus as id tensor
ntr = int(0.9 * len(data)) # 90/10 train/val split point
tr, va = data[:ntr], data[ntr:] # train and val id tensors
print(f'src={SRC} chars={len(text)} vocab={V} train={len(tr)} val={len(va)}') # dataset card
print(f'vocab sample: {chars[:20]}...') # show what the tokens look like
print(f'uniform baseline loss=ln(V)={math.log(V):.4f} (any trained loss must beat this)') # baseline: random guessing
# ---- batching: random (x,y) pairs with y = x shifted by one ----
CTX, BATCH = 48, 24 # context length and batch size: 24*48=1152 tok/iter keeps CPU ~1min
def batch(split, n=BATCH): # sample n random windows from a split
    src = tr if split == 'train' else va # pick corpus
    ix = torch.randint(0, len(src) - CTX - 1, (n,)) # random start offsets
    x = torch.stack([src[i:i + CTX] for i in ix]) # inputs: CTX ids
    y = torch.stack([src[i + 1:i + CTX + 1] for i in ix]) # targets: same window shifted +1
    return x, y # (B,T) pair
xb, yb = batch('train') # smoke-test one batch
print(f'batch x{tuple(xb.shape)} y{tuple(yb.shape)} example: {decode(xb[0][:40].tolist())!r}') # shapes + decoded peek


In [ ]:
# ---- model: TinyGPT from embeddings + MHA blocks + LM head ----
class Block(torch.nn.Module): # one pre-norm transformer block (GPT-2 style)
    def __init__(self, d, h): # build block: dim d, heads h
        super().__init__() # required init
        self.ln1 = torch.nn.LayerNorm(d) # first norm: stabilizes attention inputs
        self.h, self.dh = h, d // h # head count and per-head dim
        self.qkv = torch.nn.Linear(d, 3 * d, bias=False) # fused QKV: one matmul, 3x out
        self.proj = torch.nn.Linear(d, d, bias=False) # attention out proj
        self.ln2 = torch.nn.LayerNorm(d) # second norm: stabilizes MLP inputs
        self.mlp = torch.nn.Sequential(torch.nn.Linear(d, 4 * d), torch.nn.GELU(), torch.nn.Linear(4 * d, d)) # widen-4x MLP: the per-position compute
    def forward(self, x): # x: (B,T,d) -> same shape
        B, T, d = x.shape # read dims
        q, k, v = self.qkv(self.ln1(x)).chunk(3, dim=-1) # norm, project, split into Q/K/V
        q = q.view(B, T, self.h, self.dh).transpose(1, 2) # (B,h,T,dh): heads dim
        k = k.view(B, T, self.h, self.dh).transpose(1, 2) # (B,h,T,dh)
        v = v.view(B, T, self.h, self.dh).transpose(1, 2) # (B,h,T,dh)
        o = F.scaled_dot_product_attention(q, k, v, is_causal=True) # fused causal attention (P10, batched)
        o = o.transpose(1, 2).reshape(B, T, d) # (B,T,d): merge heads
        x = x + self.proj(o) # residual 1: gradient highway around attention
        return x + self.mlp(self.ln2(x)) # residual 2 around MLP
class TinyGPT(torch.nn.Module): # full LM: embeddings + N blocks + head
    def __init__(self, V, d=96, h=4, L=4, T=48, pos=True): # sizes; pos=False ablates positions
        super().__init__() # required init
        self.T = T # max context (positional table length)
        self.tok = torch.nn.Embedding(V, d) # token table: V*d params
        self.pos = torch.nn.Embedding(T, d) if pos else None # learned positions (None = ablation)
        self.blocks = torch.nn.Sequential(*[Block(d, h) for _ in range(L)]) # stack of L blocks
        self.ln = torch.nn.LayerNorm(d) # final norm before head
        self.head = torch.nn.Linear(d, V, bias=False) # LM head: d -> vocab logits
    def forward(self, x, y=None): # x: (B,T) ids; y: optional targets -> loss
        B, T = x.shape # read batch/seq dims
        h = self.tok(x) # (B,T,d): look up token vectors
        if self.pos is not None: # position branch (skipped in ablation)
            h = h + self.pos(torch.arange(T))[None, :, :] # add position vectors (broadcast over batch)
        h = self.blocks(h) # run transformer stack
        logits = self.head(self.ln(h)) # (B,T,V): next-token scores
        if y is None: # inference path
            return logits, None # no loss
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1)) # CE over all positions
        return logits, loss # return both (logits for sampling, loss for stepping)
def count_params(m): # count trainable params
    return sum(p.numel() for p in m.parameters()) # single source of truth (reused in P12)
torch.manual_seed(SEED) # reseed: identical init every run
gpt = TinyGPT(V) # default: d=96 h=4 L=4 T=48 with positions
P = count_params(gpt) # total params (~0.45M: check the print)
print(f'params={P} ({P/1e6:.3f}M)') # headline size number
with torch.no_grad(): # no-grad probe: shapes only
    lg, ls = gpt(xb, yb) # forward on the smoke batch
print(f'logits {tuple(lg.shape)} init-loss={ls.item():.4f} (should be near ln(V)={math.log(V):.4f})') # untrained ~= uniform
assert lg.shape == (BATCH, CTX, V) # hard gate: head outputs one logit per vocab item


In [ ]:
# ---- training loop: AdamW on next-token CE + timed sampling ----
ITERS, LR = 300, 3e-4 # 300 iters x 1152 tok: ~1min CPU; 3e-4 AdamW is the GPT default scale
EVAL_EVERY = 50 # eval + log cadence
def evalloss(m, n=20): # mean val loss over n batches (smooths sampling noise)
    m.eval() # eval mode (no dropout anyway; correct habit)
    with torch.no_grad(): # no grads: faster + less memory
        ls = [m(*batch('val'))[1].item() for _ in range(n)] # n val batches
    m.train() # back to train mode
    return float(np.mean(ls)) # mean loss
opt = torch.optim.AdamW(gpt.parameters(), lr=LR) # AdamW: decoupled weight decay suits transformers
gpt.train() # ensure train mode
hist = [] # (iter, train_loss, val_loss) rows
t0 = time.perf_counter() # start wall clock
for it in range(1, ITERS + 1): # main loop
    x, y = batch('train') # sample fresh random windows (fresh data each step)
    _, loss = gpt(x, y) # forward: LM logits + CE
    opt.zero_grad() # clear stale grads (else they accumulate)
    loss.backward() # backprop through head + blocks + embeddings
    opt.step() # AdamW update
    if it % EVAL_EVERY == 0 or it == 1: # log cadence (+ first step)
        vl = evalloss(gpt, n=10) # cheap val probe during training
        hist.append((it, round(loss.item(), 4), round(vl, 4))) # store row
        print(f'it {it}/{ITERS} train={loss.item():.4f} val={vl:.4f}') # progress line
dt = time.perf_counter() - t0 # elapsed wall time for the whole loop
toks = ITERS * BATCH * CTX # total training tokens seen
print(f'trained {toks} tokens in {dt:.1f}s -> {toks/dt:.0f} tok/s') # throughput line
final_val = evalloss(gpt, n=20) # careful final val (more batches)
ppl = math.exp(final_val) # perplexity = exp(CE): avg branching factor
print(f'final train={hist[-1][1]:.4f} val={final_val:.4f} ppl={ppl:.1f} (uniform-ppl={V})') # headline metrics
assert final_val < math.log(V) - 0.15, 'model did not beat uniform baseline' # hard gate: must actually learn
with open('./results/loss.csv', 'w', newline='') as f: # save loss history
    w = csv.writer(f) # writer handle
    w.writerow(['iter', 'train_loss', 'val_loss']) # header
    w.writerows(hist) # one row per EVAL_EVERY
    w.writerow(['FINAL', '', f'{final_val:.4f}']) # final val marker row
fig, ax = plt.subplots(figsize=(6, 4)) # loss-curve figure
xs, trs, vls = zip(*hist) # unzip history
ax.plot(xs, trs, 'o-', label='train') # train curve
ax.plot(xs, vls, 's-', label='val') # val curve
ax.axhline(math.log(V), color='red', linestyle='--', label='uniform baseline') # baseline to beat
ax.set_xlabel('iter') # x label
ax.set_ylabel('cross-entropy') # y label
ax.set_title(f'TinyGPT train/val (final val={final_val:.3f}, ppl={ppl:.1f})') # title carries metrics
ax.legend() # show legend
fig.tight_layout() # avoid clipping
fig.savefig('./results/curves.png', dpi=100) # save curve
@torch.no_grad() # sampling needs no grads
def generate(m, prompt, n=200, temp=1.0): # autoregressive sampler: feed-own-output loop
    ids = torch.tensor([encode(prompt)], dtype=torch.long) # encode prompt -> (1,L)
    for _ in range(n): # generate n new chars
        lg, _ = m(ids[:, -CTX:], None) # forward on last CTX ids (sliding window)
        probs = F.softmax(lg[0, -1] / temp, dim=-1) # temper the final-position distribution
        ids = torch.cat([ids, torch.multinomial(probs, 1).unsqueeze(0)], dim=1) # sample + append
    return decode(ids[0].tolist()) # decode full string
torch.manual_seed(123) # fixed sampling seed: reproducible excerpts
s_greedy = generate(gpt, 'To be,', n=200, temp=0.7) # low-temp: sharper, verse-like
s_hot = generate(gpt, 'To be,', n=200, temp=1.2) # high-temp: wilder
print('--- sample t=0.7 ---') # header
print(s_greedy) # show greedy-ish sample
with open('./results/samples.txt', 'w') as f: # save samples for README quoting
    f.write('PROMPT: To be,\n') # record prompt
    f.write(f'SRC: {SRC}\n') # record data provenance
    f.write(f'FINAL val={final_val:.4f} ppl={ppl:.1f}\n') # record metrics
    f.write('--- t=0.7 ---\n' + s_greedy + '\n--- t=1.2 ---\n' + s_hot + '\n') # both samples
print('saved ./results/loss.csv ./results/curves.png ./results/samples.txt') # confirm artifacts


In [ ]:
# ---- experiments: no-position ablation + context-window probe ----
torch.manual_seed(SEED) # same init family as main run
gpt_nopos = TinyGPT(V, pos=False) # clone config minus positional table
opt2 = torch.optim.AdamW(gpt_nopos.parameters(), lr=LR) # fresh optimizer for the ablation
gpt_nopos.train() # train mode
for it in range(1, 101): # short 100-iter run: enough to separate curves
    x, y = batch('train') # same batch sampler (same distribution)
    _, loss = gpt_nopos(x, y) # forward
    opt2.zero_grad() # clear grads
    loss.backward() # backward
    opt2.step() # update
vl_nopos = evalloss(gpt_nopos, n=20) # careful val for ablation
print(f'ablation val: with-pos={final_val:.4f} vs no-pos={vl_nopos:.4f} (gap>0 means order matters)') # verdict line
with open('./results/ablation.csv', 'w', newline='') as f: # save ablation table
    w = csv.writer(f) # writer handle
    w.writerow(['model', 'val_loss', 'iters']) # header
    w.writerow(['with-pos', f'{final_val:.4f}', ITERS]) # main run row
    w.writerow(['no-pos', f'{vl_nopos:.4f}', 100]) # ablation row (fewer iters: note it)
# context probe: next-token entropy vs window size, averaged over random val windows
gpt.eval() # eval mode for clean probe
with torch.no_grad(): # no grads
    for ctxw in [8, 16, 32, 48]: # window sizes: does more context sharpen prediction?
        ents = [] # per-window entropies
        for _ in range(30): # average over 30 random val windows (single phrases are noisy)
            i = int(torch.randint(0, len(va) - 49, (1,))) # random val offset
            seg = va[i:i + 48].unsqueeze(0) # 48-token val segment
            lg, _ = gpt(seg[:, -ctxw:], None) # forward on last ctxw tokens
            p = F.softmax(lg[0, -1], dim=-1) # next-token distribution
            ents.append(float(-(p * p.log()).sum())) # entropy in nats (lower = more confident)
        print(f'ctx-window={ctxw:>2} mean next-token entropy={float(np.mean(ents)):.3f} nats (n=30 val windows)') # lower = more confident
print('saved ./results/ablation.csv') # confirm artifact


## What to write in your README (7 questions)

1. What did I build? 2. How do tokenizer / batching / blocks / LM loss / sampling work?
3. What final val loss + perplexity + tok/s? 4. What surprised you?
5. Bottleneck? (hint: CPU matmuls per token×param, attention O(T²)).
6. What if no positions? different temperature? (use `ablation.csv` + `samples.txt`).
7. Next? (scale to 10M→50M in P12 accounting, then fine-tune in Session 5).
